# FEVER — RoBERTa last 6 blocks
**Select T4 GPU, then Runtime → Run all. Training is already enabled.**

This notebook continues fine-tuning our existing FEVER RoBERTa checkpoint on
116,356 training claims. It uses 14,534 separate validation claims for checkpoint
selection. The 19,998 official-development claims never enter training or checkpoint selection.

This run unfreezes the classification head and the last 6 Transformer blocks.
It starts from the same checkpoint as the last-3 run (69.42% validation macro-F1).

Training is substantial and may require multiple Colab sessions. After a session
pauses, use Run all again to resume; completed conditions are skipped. Drive stores
optimizer checkpoints, progress logs, validation metrics and final models.

The original bundle and `FEVER_Training_Data.zip` are already in your FEVER_Project
Drive folder. No manual path entry or dataset upload is needed.

## 1. Check GPU

In [ ]:
import sys, subprocess
from pathlib import Path
import torch
print("Python:", sys.version.split()[0])
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

## 2. Install runtime libraries
Libraries install only in Colab. The original RoBERTa weights are in the uploaded bundle;
this notebook does not download Qwen, Llama or embedding models.

In [ ]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/fever_hf_cache"
packages = ["transformers==4.57.6", "accelerate>=1.10,<2", "bitsandbytes>=0.48,<0.49",
            "faiss-cpu==1.12.0", "scikit-learn>=1.6,<2", "requests>=2.32,<3", "librosa>=0.11,<0.12"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.")

## 3. Connect Drive and verify the original bundle
Approve the normal Drive sign-in prompt. The original files/results stay intact.
The updated training runtime is embedded in this notebook and verified before use.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, json, sys

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/FEVER_Project")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
locations = [DRIVE_PROJECT / "FEVER_Colab_Bundle.zip",
             Path("/content/drive/MyDrive/FEVER_Colab_Bundle.zip")]
archive = next((p for p in locations if p.is_file()), None)
if archive is None:
    raise FileNotFoundError("The FEVER_Project folder is ready in My Drive. Upload FEVER_Colab_Bundle.zip into it, wait for the upload to finish, then rerun this cell. No path needs typing.")
print("Found your data:", archive)
PROJECT = Path("/content/fever_colab_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    # Refuse traversal and links in an uploaded ZIP.
    for info in z.infolist():
        target = (PROJECT / info.filename).resolve()
        target.relative_to(PROJECT.resolve())
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlink rejected")
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
import fever_colab_runtime as fever
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print("Project ready:", PROJECT)
# Install the debug runtime embedded in this notebook; the original Drive ZIP is unchanged.
import base64, zlib, hashlib, importlib
patch_source = zlib.decompress(base64.b64decode('eJy9fety20bW4H89BZypnQZsCKLsxBuTQaYcWc74G99WtjO7xbDwgSQoISIBDgBKVjyq2qfZB9sn2XPrC0CQUjyzX2rGIoC+9+lzP6e/+eabk3KZTr10nq6brPIWZeVln/O6yYtzb7FZLg/Tqslny8w7e/5z5L0tveuyuvSg1Ly8LpZlOq+9svDy1bqsmujg4F2ReT+//+Stynm29NLGS70mX2UjKLHaNOkUGlpn1eFsmeYrb3aRzS7XZV401MiLKr/KIu/jRV578L/UK7Lrg7++PPr2cJo3MCqoCE0VTegVJTZcZetleuOVC6+5yLwsrZY5zODdcpmuUu/7n7xVltabKsMq0cE333xzsKhK6LRcLrNZk5dFLcP2TspNAZPn7/O0yXDI+qN+DvGf38siO5AP57PwIq0vlvk05DfwK1plDSxlk4a/1WURrtLmIizrsEqLebkKK2gkW60X+ZJb4w7XUAiq6v7ew+PB6+c/nb7+EI/Vh0/v3787+/hBhers9OWnj6f46+27j97p23effv6r9+rty3dqcvDm3Qss/0X94zor1FD9D/hzhP88jr47fPLT4auibqrNrIHKtDxQBEd6SA9Hr/HfwyfR41bR24ODebbw6os0wRH7OM5geOBdxDLtCD49/u6pHxx413lzQSPnUlG5zgpfVVMVeGntLaAWQRZMEmAFltqHHqfzdLiIqiyd+8eDx98+xH+CcKpUMLyINmtcd38KbVdZs6kK7yK6yD7P8/OsbqBHGll+XqTwLfPL6W/BUJdrDw73IZpvVusaS4U1LHFymd3U8cdqk4VZgRCSpPUsz+OX6bLOgigrZgC8fhBsdYhjTaryupa1kB7H1AedBb8OaKI1TtRZD6rZZJ+hpaheL3OAlSKr/cDLYRoRLHi+9oMJdVKU1Spd5r9n0JbuQXkq+g3OiV9Hs7TOFuVyrhuCcUq1a9+U1zAb4VsNt9GmmQVRXpcL7EHP6bqC/UjKYsYbHF6ly02G2wwH5v1muszrCzgyq/UyazLvPz68ewsndXkDmCC7gsNGR3CWwWHUZebeusrmOR2wo1lZ4KQjPHwE57FdkxH+G63TCo/n6nKeVz4/6K1BLJSUl/QEYAArRTXoPSwdwhS8c9aePjsrHTyIaTbDKs3rzPsFf59WVVn56lSjOIuWEM8t0hkc+HyxyKp66KlHsDE8WOhf4PDAwyMcU18I9UmRrjJfReoRvcKnRyoC8J9D80ofDawjh+LaORMGOH0aaLjoA8jRIlosN/WFH4zKOlrUN8XMh1dwJIsSd9+Dt7IPPvYT0ohpcxdVlgEgIbxkVbiuyqYE9IdLx694O/g39EN/794L2rmYSx8p3WqEk1FfuU+6kZ6tei+f9L5E3llWbwA/88IC5i+r/Dwv0qVXZw1uah0ifZpdlGWNkPn29O9etSk83JoIdySDVR124d4sjkE4Fr3Yb7ioAPf54iaZboo5YMWqLBucI/7l5aQ3o1XszJjeHalVWuQLQCe8VC2sEBwQ3sAxhoxzEIOsxgr3uVaTCEa7sqtpkDI1jLVgEblezxL+RENlcgtL5y1SqDpHAKeaOPjrOrbYjUeLZOwoA8BM8mK9aWoa9VIFoypb1D2zs+WBoYA17p8lDB4A18d+YMjHz549+x53C999qcYqn6sJrUSF88dSt26xL3gipVjQKfcghv33cXBB74lfA9UH9JQul97xs5AaXCzyWQ6AMwdktizXyCl4r17UiscpbAE1GdEBrQlY9fsvy+FT+I+GscRhMNW+7ev+U5HpAZheaZkAcUJrusu0uPEVvVc0L5yzOoeT5jxmV/kcaFQmr1qL0DtzHHqK+NhbZATQtYeIOYU6vFNHaQEcFRWhgdAm4o7FXyw1glUnrm174QElQq9FPwDRp2SdL8vGABA0318YPuRzGke7xgEe7Toe3zWcz7f0+zP+HlPPSM8mE1pbbGI8mPyZ/h5PcC1xues/m+la0o1l+tbyre5/Liws0WCvBJywTNc40HUF7KyvfkEkkcNuM5oYapDL7GZQA/VI9gC6BJQLdHvk0dR5TQGbOYsCKLBGNqAhNCZoCtcwRBBldg2PZHKxSJryEo4U4oumuiH+C3nN87I8X2bRjHh+YTg3dVbh6hsSZ95E53Cg1F9fJh/f/e30LUHG51m2brxT+gMj0gwHkKCsuMorwAndOsw4rUrAtpo5hhO1LmHM/4ATBSciqQCka2gtfgtsClEnHOvF5hzQ+jnQ5Cy52Jjh/nXxfJ3zrPJiUcb0DNwQSRwJvtLNS6Pb/YSEVM1wmMbRgsWd5euZMhJvzVKcwdEF7kqA44SOkZfOZlldA3LFYQAv4L0EqCIWG+jSlI6dSEdc8KbcVHW2xMMP2Gk+9/TiISCydPYhm8FC1yR/pZumBPYtn3nLfAbsQkbtrJsUUELk0VC85madIXrHvz6wtAlxKUkS8MriMsORAjCOgRz4PmxZ2gBirUOFL1U4CPB0DBxe1seFjeocGEIgsPh1PBHutSJWBIkrMD01UmVgh+p0AcxgAew2YBRzLGhjlN4GFXKjF2morrP8/KIxQqWXroHofs5hotnyRoUVIMm5j4M7evzw4ZNB+DgI1c/5T94UmOEq81hErHg7vH9sUvjxO50Z56R8Udi/GnZGMbSjsKNO9EiS6U0DBHiIfd8SMAugI7Gg8yVw2ZTV7AK53NllCnQ4/rIebouFESAK7NRf89quCVMpqguyGZz5okb2HErBI27sMkOkAA8wwRoghIcTqgK4RlgYBYziZd4cLkH8hQVVQNrr+nC23qjJrTNxPSo11L9CdQ6FhtRzNNvM6bAnQAgBqJinHdD+OgXyOkmvgHVAdhlEF2SjCJRChZ91WzJDqsILRnxHgs8965UvSJrf2U3vSfuQoQivX3o/eicXaXGeGQBAuIe3H79FXUSITCLgyQyZQEDsmzWdI1RueCfvPxFvmAP+bUACyOaRg8Sh8lCFe5coVL+cPX/jASgONZT2lwdwXmcgYgAPMQiiBsjtEhDQqqxuBKKPA40rYS/rrGepzmeRKC985hZ2L5rzBSSC5iaZpbALUO3g7NPbj6/enCZnp7+8+vAKxLnYU0QODufZdHN+ePWdOjgA0gSICZb3TVnk0BaLgyfAHpSowKnK8wox1xrEEm++qUh+wunRiUPOpjwHwlaQiAi4Dilh7ciFdXRw8BEY9ynA4jktGmwRMgS88LQXsJsZkNQaSChJst6rBneJNT+MFS+zbH0IxPEqY/WOh8uXAAnImyTxEaWGIvkAuzPPseeQFj40wmo8ABxEBBZ+XcAZaqbAIiV1hjXq+PEgrC/K60TPmCnFBjqIhftghly2ieYA+BFeYe9Rn5DlfLiP1MsNmfHH5hc3JEAb0xLBBiySmTCngSlwniW6mFtHt70961X62V8A7mv8rW9BGA2+0023Vqb1pNtelrPL2CxLdPYanu3IynUCEFI0TonTK8KqXAIhi8AlHk/4zTStYiZe3H4NfSGzCNuHggGtAv6TFMipj2QM5wkJrM7KH/lKVwUhv90SUGzDdUovDQxvFbdaE2H+swpJXQRjj9WmWRx+71YDxB1/UbZpoCLtvgBz6v1UQwukiqAUESoCqzLQimXkZ6gEbhMpK48h9O7+p3AUG8D7CtAcspNIMhAm4A0BAr8iYKYRMkLXQyyy64T5VDUcQDFYCGSkNs1MDUnjFCo8DWqI/95q2dSBBDwemgMVHrT5x3wVIRdjzg28oM9mj/GNTxOLeQ3wTIPIFNv5z7N6Zk8Dn0n8J5zfAGrOZwlsy7I2x0izca+oSyIjQAotpDIIOpD4kX75sESAxHnvE3McwnkKqLsQdYjTAB8u38AAwbcFNjl6KrRAiWyXC1jILBEeA/JMXwjD6BVsEwRvPxEw1B9E3nJGG3eeT4H/6ZIppkOJKeYHDpcFu5hVV3fV1aWcqggOW/KCrLkMjc8yTbdOrwhPAbDaafNz/OXhQ3uiAOJIN9yGQr3CQjM0vjID3saPhy4yxJketE8NoM2vaMwiW1o82ESh8nL2aVtpaUiN2MFJNN1elOQoFFmZyPorUqs4Km5uAY7LHPHq4z514jbCGnUViO6oZFCizDGbdcP0FfvXOikkzduEYajVph6qvGM1Vo9oy8bHx8PjZ5NHyvv08WTiyZzNgYb3/0QBBtoXrCK4QTMBCDpD/ZJXw8cecIFIwcc8HL4KSX2qUQGx8HQsaQa0Z+HDh8CiLIG+4VxIsWjo11DOmUMKotmyBLiwU6Pv2NCD2AIqS8KMbIOhRXBtktwDTG6DsTAaBnUIIgXZ4TK7YenBMlYKycJ607D4imJCuWlaz1WKWwMsaXYNT5k+l2oydAa+Ltc+tB6SOH7gufRsLBOaxPR35NQSk40sJFar4NxUcyCBfSf1LnLYhgeXRHWX2HyB1mlnoUhrV2+dOdD+Mbg454ZH2ndcHqlfAe5Hbm3RxptGW7iLX8MJiL9cDq94o8Ir3Cgei1bk4g5fEiTniKKZm9y9P7e3bnc3vsza+xMbKdwNMssxCfgUOdDbs2bugWKZTqngES0gVFePXOwC0+pbIw2xMkPbAcC9HbI6wVeoluByRoE3Hh5/P5j0NeKuRqups/TaY9D25Lvbrlutt127rq1WmS1wGnLOh2AO3Oq54L4KECaTKWGc6c1YXQMhNTRj0uFlo3SNuNzXDPVB+3RZXm/yKD4etQ7eNk+GZToNtNlCKIBYUAbGiuUJ4JRXb395/vrVCyInqywtSA/UHuj48LvBcBIcoWa+98uoylasoozdAUjHhztmhei5SWNT9yF2r60C7X6CH+MnVstwHzIgKOg4MOICYJcmWZd1s8g/+4go4/ZaGPFvzxqidhVI2YVAe4e3E/SMcKFCi4q7/ViRc8fCsGjat5TEmbTGtHe0MiCBSdGc8RPyJIDqYNwJLnunEL7SjBC8Ob9IzC4lF6iklHKwf0dPng6Qx8MtgcdtHEJiBI/b0h9eEz6CLToVhC065ZZrEzAZnD3fblHYozWADSrLVMBIJTQn2C2Y4TmXtuyOWYbcKdr5TE0bFt3IA5Zjvb7Il5nDChkhN7pOcy63LXEzd9DDdtzJd3h/8v6WZWsyg6YwzKuM2YaRl3qmG707Rm9Dlo8S6lSiiZ4BxoqoPWF54z/E6R4HLnfTT2HbTICQaWXGqO5iDO5D/Q9c7n27pLBi4S7uXl7c7qH5ox0E36HM6kOTL5fkJoUOBv/0dhDq7uiFWD/qHzXCnS4AzchQ8RWp55uLXLRmmkcnUDEsLkpRap1uakRSMKHaMbZsQSqaUdtSLfmeIDDAaYwfB1/HI3eYSNRKEBcJP0Z9wjJQSRytjF4kGxk7/wm6fB1POnBHs5NW6LLiPdPkixtfn5AeqQDdGvT5cdvTL/1OFW43L2q0+jJ58GH1HFWk3nraCNYzAqHJKrTtMqkJyanAfkaVtRAh1sieoc6U2gZeu94sm9pLF+jGlxfS0MgzbWpDI6tlMzzOcvwJF7OTjquyhOEGRw4GrNWRGT2789TxeG3tGKLVPF+WU189FImRFo7cYh7EHV8RNsvqZcBVpRW10iR1IYKcelu6GmSZ9U3WRN4J2hdgKqSqxjPHsOLBbMvriBwWHG0D+yGhjpOaD0HQidkbzVsP1wSg6FsFFFI0idpjxPV68fEQ6pEHojLEo9fE+xxdDjzgs1ZpdYPCQUPn+5INQZcsygEiCI0swKxDqNiyqZ9gC+q8bhB/hsYXQJDR3fJfU20K0vGorqZw6z+FKGENCJKN7miGa7O2oTrPClH6Oy+ZuqL5SSsDDQjK3iJYm+3u9fjBaRFRA4j5DU09VAVVhnygtqCa1a68uCAA6K+JZnlddxXrpXM/p5Ux8QTidDJx+1mlzewiq+0EEaPpj4bb3j0sbWlyZSwuvFeFE4zkTLw3B8JDI+xQie+XZ+DCOt3xaTbOI0ea9jBB0dNzQFnvHmId3Djp89fi9H8+P/novT979+b9RyCUjxrNKgEzi0B5Vv50evYR/XPR2A2nlGCS7NGaUeMP7FiEDq3ILxEHTcyV7erlp9evvbPnf/fOTj+8f/f2w6nTn2X3nOWzY5mm03yZo8ENSL5eTW4emVZ3gVq+auaTCzPt4i5wjMxY0VX39avTF9775x8+PP/59AOO1RmabkPI+ma9XuawBmZHgv17LkSt0wof+wBw69np8xf/Sw3tLv3y6sXp25NT7+XzV68/nZ16L04/wq9do7qrc+0Ux+DJ1E1QruaE2uRtmQIOJZJex993XQ4ZGKW+Q1baroWu6vHAJQp2wyxxcPE/cxDQI5oWAb0qO64ONRDuYh/SHm2f0bu1rLBiaASN0d8YKH+LNGoG56E2NN2LBqGIDC0aV54eP/Y5erpo5gwLjw+PJ2ypEp9TOHfLeAlr51NZfxECIVyi34vdrfYJPDs9OX370Tv9Bf4l2FHiioxNWfB/idZ06HJItg3q2AEaWq8DY1fOaoS55nQ1zeawLJqXAe7UewPi5us3HruDADczq0qooV2ljKfn7GJTEHuNPjNXsAzopwKyxOwi2mEJ1h4YLYOtmFHYLOX4XugCzzdN+RFJZ/57VoX49AZpsTEaYf2Y/hV2WZeNWzUj7AAlVvLpylDS5sGM2SNlYp2V3C/inzIJgWjD7ogTFTqnW03En7wPSJs9dlqvcKlvKJjiBB0f0EKOq8T2cvJUQJdg4jiQutJ+oHW+ivSciNmIzVT/fw09QprroyME0KH1hu0aZFVChU2xWSUsetT+Y62zZ7d8Y3aoQ2N7NpKMbBv5xiDAF2uyNSFnHI8nEoRAUis5L6LPiD8IUfNELQbhk8dMA6ZI2GN6OabyQ/r30ZPHE8HD6DpoVFZm33XwwOcwnYNkCaxKjr6FrNUgA0Tw4+PvnlovReqo12uTDgduHnkmoh4jm9cIqr9nhQdtHFKr3jJf5Y1iEyf1LQomMySfugDGgFoTXzfmAnHXRNCgg5qIxxYw6tKkFfN4c4xckawoRoK1FBc5HOMitvDjP3wogwkiwixcgghFNlql9WUs38cqbZCXRXyI79Uk2hQ1oCb0Wz8UxcJV7HMDD7EIIMbNyj8OjvBBfoM0l67W/iovYqnDu261rWYKRREtYPbsqRJZh9KrcA2ofJ6voIWIIAjAE8aOLnSK3CGePFaGDBM4xPHA0wAFoPFjbCEJ3/v04ejoyeNHx8F/Ox5A8WFX0FSZ3ueE9kmF/AK2cYXxHtK0A6MBKwqJOtVOjzZEAcAeDSfI5heoDuWlEIlUQjISLYoRIxyaUfS4sjtuZIZXGSpyn0LfG+/v+WUOYkKeehIqBoh6kVd1A//C53JhA8pAoDhsMjp5l8AXfjp7DWURfUnNI0HlqDyp1+ksq8nYWVbAcqDAYldLDc1PEImQGiRsqteizvDxYBAqbi+ZAZlAH1p0gHsyIBmKFmF2kVbpDAThZLqZQ2U1/O+o14SOWIziGCI1bLvb7/SMD1VdbiqUxqTel6Jbta5m6qhgma9gmW+B8nhCPliJOLBF5Nq3yAAgkmu9vIleXv6arS8yoFUwiCo9x1eT29vWDkMX6GZPSgTUr2rIquxub6NQIoW7Otb41ez4c/7wEstnldRuDUxXkSYS2qo6JFPQL7Q5L2AxpyngIPRMTGTw2/EUqHkyrOOIoDe+C5ZHEv8CVRzJh0M9kKOiPSZWJG9uYhvv8QVloSGsHQtFWiLnN2K0QlMdArFZHPqszVbtL+F4ArChR4m+pK3IkuA2kOio9kA7IpoZT0dI25JgSJK6Qw9BOAzLjVV7GdTkQdx+00OcTtCbcG5kEV7JIz1Bb5XXJBkTDdnGeFqWIp9EIA0Niq11TV6zRFJxXLukpPDLLaDAjt4jlqlotaKDC+nLgfbu7HWUY5gkJzM6vvSTtBj0az6l0zEySElc0MhKEN8PVjprPGw/h9ZL6/Ts7N2ZujVe+yZ6DHFI9FEjwxd5RYcH1zVb5J9jxiIclHDIEWY4XjYsmIHTAcJHOBp8ZuMdZ9nnzZnnVWyqHykD0UHbrQsNCnLAUSj6jB4kHOwCGC4DYkpOzP7eAzIefjcJj4Wj2IYaGi+SSamkQvkRt3tm4qhbJiL6XSgRR7s7B5DaVGRWo2GPFTyJ6dgzU7vDZcNZEFMnlmWO6K/fbX+sV5TsomRHYAjUfMuXy6GUGF9OXEUhgA2uAbUDqAXtGaQ71Kx354mLmUecAcxmtaaXTZVnV+JPFSodB6mJ2ORWZsdCVw2opsGhtTG6fjRExjKgga6/tae6CV4kGDDNSm+onuQkNBvQMlf2Gac6W4XWTB5fUpfJIq0Iv/AbMywKxeB6/KY34uPzTG8tYwazQ7QFw87Oai3okCMwPs9sDMYjpSNL8TVA/WBgl3j/EiUcuKf6YNWxdPIAUQQfa2Ustb9A0Xl5M9Qw6VhLxFjNvRjXhT5IP7Tuw1ojw+vZGy3VTiFgdFfof0+be4jBxMgrY4zboaEnCxgnOodHjGj2MM3Go1KQuLu/o/UF8BW7jiuzhnVsAFZEt/HnsdM+EkyJSCR5jduePBq7+H1i+Sj9IxjJkjpD3busNNbeyQoPy+oNO+GthvWx2NkvnIY23gbK1sOI+Q7Cn8s7vb6hrNp4iAqe0Tq9QQ4jnOUSPRi7TJzfooHzacRT8HUbh8fDSfhkIAdRA5Nuqh+eNEPgaY4BoLohXQZQQXgtXXvCz3PTbM8X5xBtDWV9aShTiPVUtjiO2IyoB8YsahLOQf6GfJKEieA/rTOopzyftqyJsEhyKjXbsSMADdBRa1bURXw3tjkeALrpG92umWBgsh1zq09gPShk0TH/7MMYuoc6AWksESBIAPuSV4wBOm0YQqyA62I+GB43dNzQY3IoCGwcEUP/vq7UpDemyPBWXWEUfqbFZi1R0xTS34kdZ0+wgz2srrgHddlWGa1mWy3wtfhhxx1lFz/ctxUtdNBTgBDCXgcZK/rTN1YgQDM12kEq2Ao8aqEcubBrLhSrOfnCdD71nO137M0HO0uiRd6A6G8VXDrOL5X0GNQibsWfvOcU/UhZKby08Mo1a3ZCm7XiTVpdYjgf7BsS7SrUJvDPOK2GslpgSyS91puKFg7VFbCRNcEhLl6OVnXd5hGpuQ7F/qnxDuaZ8abl/IY0hzqtBxwg3Kl5XGURkkMSjfxK/ed//qf/lyGKacFfxt6vzeThr9Vffi386OFfAvoFBVSIzYVQ88W7j89fv/4n/Hr189t3Z6cnzz+cMthz60Pql39HGFS19lETtnblQCxijwqwgkVNYaP+OkQzIimt0PljHWjHVB9DB2Gn1W2of2m79C3ruJxo9TVp49fG8qkb4bD4QB9ot9+xNDoJ0Shhitj3PZDyiv3dSJm0xMB9j8p6NQDOKlWSSQHdbGlQtinrWOAMAcu1Z49vggcxz1ch/a3wu2ttz8nQ/dumhvOesyJV3e4ZKrbkjE/rj3uGAmw+HKTKLha/07DkigAyvPYoJlSTapmBdzbirnES/YRGdHUaMOATbrS1BpP+JZ3Xdjv75mlmmOuRCQ4Z9yA4Zr5yyhw0Z8Uq690b7AjzMeAj/twzMcNZSnYHHHVnkR7EW3mVCJ5phPM+duQDjJXMe0cgg28o9Q2uX0GK+tRwMtif9FLH49zdFzMzC6UTtHZgIqx5bDI+4Widh8I0J/7aJkkUAdZy6X+OY+elYV51NYnd3Zqvo+nlIzyUobR9UIZrphL8hJotGvrQOc3Wt8T1V9HNxbGDJHBuLg5pu49KI1RAWBM1VNJOQsYpyY9AVJMRlDPYZJXXNemLdScCcWu3ee1KkHBqJMyOJTQjIUWbRbOycvT2Vhs0T9JNnS4l5QOaM/vNj6HOFfOv2h9flhV3+fpN+BNQyufF/CcMCz8pi0WO0aduxPWo31SJAyKTHT+y80bc8o2h5Uz4S9SAHNRyi7FG0PvbPPuNhm2DYV8ahn0W0I4Bbo0DpDa6lrmy5g9QhxIExNtL51O/eZF8O80bNpVNiyk9JVQnQQ47VsXiW2W/bABw5uVmCswzN9yuiDwDYIdkTnXZCEX2pOOnZvwdq6uzwf8l6xi6GRNQfIO1iOldKOHrq3QNFFENB7ccD987GzHqMvyLsx6DP9ohNJ+4Sj9TDINYRb87fkyv5HP87eDZ0z7T7sIhEsIvW6+Xvqw3gvEdzldY+JFEkb89feW4FCp2e6Bj4BwJ9AX+tXj19v2nj8QoDjv+M2ZQIkCiJmbbHYTCLISlH39RVblENIZ5VpRYoxA9co+3E9JeF4Dktoy5qHVa3qDdCkARBJYlymG65VAXk01FE7TjoSdz0xGxO6zFuuM+AzY3ey/jMRv2MS0E9kWyIEWiGLOv1oBjbqv6Il1nY/IY2ZaheNgJYp4bzBjihAlQs6EJUG3HELTBTENfQuJE7ABcoIEL2nrUrvSjW459w0jMRg9FGFPXKN/WPSCUeTWmRkGpVC+Z6lXxmDn+Rio91VXmSTRdmzabIB5dXwMQ6nR5ExP9wexq2z90yCvY8RLFSG8moRgXgDGHvM9ap0l/bvv2SkNbcZ5Y77ztLQtGruPoXq36vRwQePiO/0GkBTPriNBFOx3wmJcgo6NsJzBtyEiSb51B9xuO0UiBMQ9kPAhpCsPJqEqvu7XnGakZTaWwvszXva4ieEjLumdaFklbqgZjGWFp+D8xt5bdhjfCjhPvMoZncmIF+SmeluWS3FjMcAJievGomlfkzEWWnLLuPaKGfcLYopw0He2z2O5hS1MQ44vAuur0gXiPq/GwT9m0DVl/6HgYsAWScn2/09KZ3Lbb9BDfUMh3dTMUvxStQnOUKdihq0shvago/3xL2sKPQHj519+yG/oRaJUgN260/rHW++3dNMM/E5urmxD2/IGNFRTW19STuLNW0FuntoPHZN4d9ZJJYkZfu5ErEpzJzBHZYDscJ70zCXOMf2E5zaom3cuPk94OcQDs3nlzgVzIv4kz/4COlXDuTsS3jvH+PZlyOyD7Uxj0dvbUP8p903wxvcMy4eRnqC8zzmB7eett3rR/iv+OPh22wTCTJpSHcaAgPnQRZJf9B/GTffke0Y8wO+Rci9oz3SaeVndxqy1ONHz4cFNgENV/IUv6J++9cDhOolNtBSmrOekzS1GAGj3HLMe5T2+gTV42RD/IXptUwmhftcatdUsLY3hZ/MnMpZTLRFth1TM4RuBuNP11OEnX+kNr2eUcgxYbuIcn3deSwokLTVDuke6cqHuwqds4kjFJYriODjdzN0cbAsXQITcuNaQVC0LzzXn5Y2fgdAQM2/MvsksYGLHLUbM8B4E4qstFg9FRh8fBeDBh2c4PohnmLYFWyXncwdyGVrNycYxV2bH2ScuvPaeux/kkIHcYM519tHxr1mEnsIPa7FLpvRvCWh+1vStMx2VbnDCp4e6NCbuBLMN+Vq9vQLi07SXFpDD/IvkzyfYItRn3ona28i8cjWcdE91UhBKu4GQux2ztmKUskEi+vOZazCeuo3qzWOSfOYkFe6WFipRFYTsJJDxOc8pj4eRQnufpeVHWTT7bnUdZcib3JRw2te+fUFlau19eZTJ/qzvTK78w47Apll0XuD+WXZmJ2r8pv/Kzwd3JlZ0y98n2SzX2EVxob07+vpRL3S4Np6wIdC5en2eBlpY9SZ1xaDrfsky7PxPzk8G98jCbUT4ZuIMzkXk1Xk/hiWbbpGSmxNJkE2oFaHY9yW7vTsbswIoTe2sTMm9snmgOJO5k+uVDDtgpMRFNPmcA1l6+BPGXeTE3IS+Ov+8qLzZNVsfffrdLDWf54b4A0Z5siL1ZE3XjvOnMO4kuBUptMFGBlh/i40FYz0pUq+rM2JgzU0mI8/MG9VwNpmyqbni1cH2F8WF6R9kiMeyIcBLKEitKSanjwEjnU3Nk0Fb2Ue47Vk4O8GcDZSJVr4G/3IWt9m6vbCxnrRdBwripsK6Bk9HrMTzoLIA+lW6q9J4uMfLqEHOYsv+7twLO2uPEHb25znUs1YaS41/pPNVuCnTnYOis1CDG8nHYsYs/HO+2vh1Kph4UaokdByhaznmgU+CGyzrHpnQHDKI/xH1I5oPsqBTa2UYL/FrZSRBCugbJdnFg8dh02X79w3FvXndaRd093lgjicEJ5ugiAuyWTjFdAhPVGRDYbx8HI+bQgCJtUoypbr3l1Hr2E+bmw893+cVLjCX5kbcC9xErwNJoX3eNGigLFeAKgD3hdUXM72MjbIAXpUJBzl+7fANmxGbUkBCPchydrYtzhtzUt4/Dr4ytOPC0mnRHzW1zVagknmJXlbqaHe0Ivwgcc7/13L8r4gBH2UasatjRNCpHpwzrgdYOI2dYBMzVNJ9JjCRbMQHzZvMbV7mMAjf5UNLtLOyqwsEyNlE1csVO1moziO4x5m57PlB7Jnd9QghLDX31bCDwrmXTw1a6eo1HWL4lRzInBb7ajX9F2dR3Q0MoaA2wmojNSDabckOOQw1yehLeiFjmqR6gvT9KIasNYznPKopejXdckzKSpIsUGUAxFWMTsa8o6QQ6CZngpnU7GL/WPktEj6DA//3f/0e1UxESt1Bea35hqJNTuEkmeqM+esM+7sw9welCJf3SHIPGTeO0Yq3EbfJRmEDnu14eG80AZZ3V7MPalKvDWZ95mTFKJh5Z20/MMeO8LhQFIl7WjU3ryHuiP8AgrUeTfJOgcGKEVLoky5HxBjPpQA50xHhs81xzpJAJS6cW6B9Dh508XiZUJbBJxDYrv+nLs8YRtyRhcY3e5BvbrNTWm3BP1mXkJsSZORa6LjLivugXQE3xYOQeefELHYx0ZjHKrnMZD8QVVHsv6oRCI0nKowPbxQCmffmUCWaRhdWJfhBQ0VRkXZWJJVYhS7zmtbZ2EwzoCXaDyH2LhHf3ZOOddScmMb+klIB+wvY7onUB5k3T2e5w4wXU6FiZte4onrcIpsNgB5ryxl3K23YmMW2EEovZnRpNo9bmUUpvf484CRsfEUhUtOAhfYQ0tthd98dYmLCHTwfDKQKI1LmD+YJPP8btMkMNQOaqDrokUIOTw1AZmBqZHv9IGFav5bSDzYYOLuvLeH27P6ygsz3t5K0255MdKb1qIduQ2UlYEItiHsFqPTpuu+9rj4mvCvhsjdNxk295+OmQQCc2UlvNR93SJqXXlk582wdfn5hIq933+Id02KcdEwh6MdiBjeZx+Op3m+bd4g1lZeaMnzTAzsZh5mZYw6Rc6ATOjjFL0R2VDJeMF2p9cogzc0jdNebdSKdVTupECWjx+hzQ8dpM6BEDprnHyHuJyRroOk16MQK2PC3wUhxWHhzpC888PDg5XuxEl2aifgB6ci976bUkuv0bY6K1LwatwCe932JavGeoQc+2cM7SHSdFRzjx34RzIOuS7QzEXdf1drBTby7W/ZE3UgXKIK0ApIXpnECInS03c1xazq/o5JnBa58uMAUbqUIdo841XjgCG7G8AVqYLxvYOEJbNsKJO9HIEUDnsCkPM2wmX2eYtNpD155idhMpZ2jdbJRs4b2DXeREDjZk4Y7iEs2AfAFmoDVrTxnsdKSDzUcqvEH78dExEY9dWWeZ0A2EXmxDyI/Gote59oUt65Lj7olbEy0kVco3lwKaOuqa2PDWs8h7JdnDUDhYlnBWOYsc96qVJhLfpzTH3J6a8QnqSkj3GDGfjnZ7mMLRnYeskfi0OGNmwbY+0vZ3I18g3gRKTLin7xZOgNyKbI7sJhVpAaJNXOLY8LjDNpdnOGfD56H8xXMxdXAaW+n5lAleAuQyLdNq/grPXLVZkzdVu5tcf3I5SvSQ85wv0B4tVFnAXDcVXpwCsh/vYi86Vtsck9tRy0GBUkqOaBt3R11tjVswlh7yXXhxezyozP06tNcZcitOi3mEbldkKzKcdm1gXBPkVjCaodJOoKjhvzu2JxvZprvkSu5ahSZrp8jQWpsn0VEAhaFC8sVcYeR9oGpDWBa3FYU3gKLCc1MQOUR1qqaDTcnidha1RW2+L3gGx8VvKcrrUCgn59akwvbepVYeIY4OuqSLttB7Gii68QZJZwCM6ewmoS7QIWbGgdbwAwnQgt4nNQcMIKez2NSc8gba+XyXVo81Yfc3IWEw7VcakEbaBYvTN7qrQ1SOYi2ot30qaXZSmDGVNLwJYl3WRmudkdbusFoftSY6fAc3B+UR2iS+PnqPZlPUM18wdVq2GrpqkNbsvj5XKV9mwpaZXQYrXhd9Cw7IjoxdD3W9/deCOjyjE6SytQG23QexaRhPpqv0ePX25N2b969PP55iPjarp1BHrDfjC9BolzToIn2Zb2ZEH0ZCUpArIkNLPK660+QkSM4yaN3GZHQTjxECx+5nx+5ILU5GlIx9zIk7WiVtaEinig02ouehM0pz4kI+ceHiOEziO86hf0Oua6FYXNlqGP6eVWUyz8WvfkAIm05p3D22vfUfjQ2bgzktnoTDSdtrghzkcJsmeDeUzvthiKymzDTFIJarb1kbihputA7BEiS0e3gPIHsocHG6MpC2Uw3bOEnGSprfGUxwcQyCLrl2FOsIE7/7i+OA1VLK6IKxRkSMsm9vDAhdLhTOx2aFSu7tyZNfe2u9ULmMf7mXNWrdUR+jhl9oBYdflNkwPTjzAh1H0N8Ct1F/5Cf+YuezOMY3bIrNeW/aWUgkHPGWh8GYnLn9oXrNV/K2/NrkwvOXOuHV4VbCKxu07s1LAtAbtN1SOKpGcRjAB1zcy9NfTs+YEkWEVJxj62ZLvxy6kNJJ/mGS/RbK2XFna83tYJi9Iziwfp1Dcig7yxD8mS+UUCpWWY+Apub/2GQgt2C/GGNpGEe+aKPc1J6A4ZEJsK24vYjgG38SmvZlbWudYrZ9O5sROEaObMLVQ8lYppNL8njxgkX64VjE2W2UqLs1NiRcrEPpxZ78AZeeneXqjloapk+8PwhrG+SvBfeyHOPxfQPsa3e+ySiqrGVSntVXdxFy8gTZYU3GSe2+V9s6KPS7fUj67/huHkc7d8r9CWjSMI4yFDErTbmeMSsnY7trqHEmkbQwGFQoUOR7NhDUtaOSjahW0g0ZlqAX4+u4y55EIdxsUcdiLcaCp6Daqk1tYnFSbW0xEvtsLv0JaB0iJKrIvblkjQHx69J9WaWf04vbZpvdyZ18f+6oeVO+3HKDFeaSirvSOkLjuD0QS5qpIiDT87K6iX3XUmrTDIjYrYwqQJy/OwpDm7uYRtU2jJjLTTgehYVidbdyodOKzFFCQg22TjBUN5GLgnixqJzArBlpWSUsunBgCxZP5EzNlcTm4EubE8i5x4o5snsllDNZxId7F198GB0/4WHnshk9YTRY0w/O/YQKd9k0+CK/bKqibv4HvHNC+pIL4mm1tE+mTQbZukirb+C2uuM3LE3biG+Qa6t1w5o55moEePfnvtCrISFUOpIXhYkhjoT8GB1rvv3ing3dDq8T9EJEAFgCZdYOeQTk1dStJql08dSBg1F8XXYXyQMioYyjbvsiUdQX0hV5SpI1G71dFUO16AVs9t/p0V+E5BxG2J3TOXO348EkEHoqd+KhepLMfO5LliGpRmBYEKCy+MKrL7KssZnkDetQDx06zXN1EPn9SL0Q9rpLwcWtbVOwJT6hnLHL9Carap9Oflg42USOf4iLH0hdxa7tYj+TBE5VRBX3ZQOgAhzxp+VNEgq5OURgMB3sOxii/Mjia3JepXOmnltVnDzKe2ujfoErT/HaFNtA7xTGh8VwMhQiRr1Rrft04Thnw0pSNYBcvkVPRoq3FwFSIh8+RSEb6DOX2MbhjGxWIEAj47z0g33LxIJzayCB3GL7dQ0GtwwRQNthJkWWyAJZzsYBkfhYOxCu82XZKOPl+GQQOn5MdZOtoexgMAjMfdZEuufUzSH0g+oK46nH5q8ZeqxoFw/bmod2SDaP8l3WOZsJ5t6zp0fPnnrkXEVcJrN7Hq0wJacvF801cN4iUdDIWzqSA53CxnMYbMnRZ5sB2R85ZeGDtJhxCHyWKSO3Yrsej+G2SuvfEd0UfqSQn4r/QtfPq/MN8nq1/nIiGd1CzDJ2Ui6XKYzn74AE33NearRck6dbz6giDiiqkk2TL80YMdcvAYHdlP1KusUxi8IH3Uisxnrf7WXf+zLndOHLOA92P/Q6SJ5YeCKtNKAn49LoeBXit/Os0r7TBDHm5kiBeUrI3JuE5uSixDRGVA5HRgWDAwYTolwkZ/gsaNDbhAqLPtF10uLONE9vy2Ob9tpsmMZWu5aN/wONO5XaPdhsXzwFy9lK3hwZguXTeV1eAu1/WzYv0THC8Z+1x4rOLcrFsr6ICNh6DzgbKi3h3H7IMg851jenpJEzx/XwUJDAYeeURt4LVgvUmykgkgaQk+d6tkVmOxwdr5lbMILJOB9kagCc0sT9FcPAYhGSiu/y9NdN3x6YGwY03mJAEtdboIalltCyIxSoibPhhcsr9BeHk4fm6IOtE0G5WbfOD8YPHcO/sHezLOfgZFqK4Oj4KV3f0lQkAN01Bap0O7q6V+ErmiqJFVj+z1c6CZLPL/5JL4I/6wXsOWYa9TlAe2RdHSWrEuDqZbp2DzLAPR1Io13nyf4QP3vadzUSHWIC0hkx1YgpOEqfzrUBZGWsLNRLtMHEZ34glB49fFgxzL2Fyg5aPsALeP1XOD1lRUTQ0qAa7w5DVQQrQ9Bi00rzefb8Z3ughMdme6q+xp3B1O79kXaANax9y5PZTAr5aykJjCZO7JEiIiDWSMsTBNoXOlpdYvgSMBd0Ew2xO4QUkvJS+2HeN4hWht8f0xoceF8dKXtXw4Y9i/ewyO7sRXNm6rVdtcV7l/4IRCSiR96GB/2FAYILb7lTW2TVqrpVTuMuFLb+NedsWTKH+oNI2+O0LuUC1wE9W5ezC7wigIZBzAJJ9inqgB9nh9+xf7T2xMZYGEW3MyQ1wAW0ESpkcXNUZqH6dbVZsj8zfuGra5J5NktByo4Gx9SJ4zuNKSlbaA/7PYeyijAgxhd1kGJrZowmlQACOYWbbQ73+Hyjz3O/nzMhI+0WYiwRWs/YvWrpuTjUCuyidVZqsknCXLDUfn3gSa4sxL3IBKIhi52yiK2LyONSPpDScTton70TKAiSY+owIEoXhN2ScibU0dzPQhVNSeAcUbepW83Ze7qKTdFxPhlhidgJQbZ6mspxeuMI6nuFI2OGAdIiUto5ubVuIlajiJKXYy+iwjGriMV7XZX18mpv5fDeqFFuGh50fIk58QkIQQ1MYo0OSr/ThUIZ0SvjjLAVAYYWBeHydWA623oNDZim1Ug3wFSp9vCWZ4rRQq+GoyXe8sTXT0UaUtB35nnzE/kS+x0xoh9AAnPl654QaayFKdAJkU2z85xhJEyrc7r3s8nIE7Eql+HDh5fX+Dro86XQs3OcO+0Vz3x967KckttCttaXO9N7BAo+xTwIgmUH+ToI1Hwi7Nsnpcfm5bhfip84k8Ze/9Cc84XX74XyILZUebhjZb5iRbpj1Xf33D1SUqjpgdCdtK2Ba8/TP75HRCzkLf3WtoCtBviKH7kRqP0p7ja7a83o69eA0s5h7rvDmfy4HTfuPr9MMzL26k+0B/a+MQbGPxk2ClHGZjmnHMiEMEadD87cy7UosEx+ASlr4WJZnu8ECbpyTxJ13OfoioZg/3pLoZgCTGFzv9w6UOrYwrHwv+FY6dPfc7CcT3etvgxP2I5s57j05O5eNAcrGZvdvdZNq2G6S4fw8C+iXpctYh+yfQOypV23WYfbqskGZx4P5crurfUlMRRHF2/pu3xJDoUXp+zqJAjpfjvC/cyKxnjBl8OFxsSEupxkLIwkek1ISkahYIYvjb8Nna8k/jsfvw97WVYRxb8NF+vjpywdoafTFU2VGdNY86XyFstv8aNyI3qnhrztrXHAIV4Ja4k5Ld9jPMHnGg3WFBoOXHsMTDuyh4l+YAM/VI3HZA2jNJfTDC/MJDth2iDl4OmIgweI+k6J2PGVOKcQjyrJayjQoLVFL4NZMTt4c4Ve/7eEoRZkrA2lQ4MCVYqsODFXtzQLHCx0hzCAue2Rhg8kC5GkCvXXNluH668jCkx/HbEhExMtryPHiB1B55K7RXsljQfhcfh4EsJSV+hUyu2pLTenW5Eyafr015cIKRIuaVZ0Thns5syox212nvkXgpPeAsTFAPM4Y/cJ9v+xXDauOe3zTBTE8Q5tsW+rcDK+BOC5yeHkJ+Ui/p6j9DDnqsZB8hzOhH+EdWmxlwEHM7ODS9yjWvZ3HuagLT25nzr2do7EIHHIhOX1xtFZW8EdAXWyY6wg99nzlZMiO7hOT0s8CnWdLbT2Y9z+ZHkVRME87LZMN+rgZTjPuKvsrSuxfKQIpsXDtwHFDXEnePg5CRF/GtnEOfTN0Y9wgf44ho7AGnbVGyJB75w2iPaIFlzy5qgU2tUcFGMLhbgaeX3RjhLjGNa9LvScyWxTuWYdGq1141faBUoTGvNJe0W3I4olRrbtXt2GHh30YCeg/amJhf4DTtV7nfv/gFs/efNzD2zAwdCXunGCGaw20W7NPRz1v9JFvwvWht3QXvr38cBvu8G3HeBHmPhFIIvjUF1Peq0+wbYO/h+I4kdr'))
expected_runtime_sha256 = '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06'
assert hashlib.sha256(patch_source).hexdigest() == expected_runtime_sha256
manifest_path = PROJECT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
if manifest['files']['src/fever_colab_runtime.py'] not in ['f2ca840db6e6d871bb42a1bd3ab84ef77613d7946541add86c407b7c4b26ad55', '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06']:
    raise RuntimeError('Unexpected bundle revision; use the original supplied ZIP.')
(PROJECT / 'src/fever_colab_runtime.py').write_bytes(patch_source)
manifest['files']['src/fever_colab_runtime.py'] = expected_runtime_sha256
manifest['notebook_patch'] = {'revision': 'colab-debug-v5', 'runtime_sha256': expected_runtime_sha256}
manifest_path.write_text(json.dumps(manifest, indent=2))
importlib.invalidate_caches()
fever = importlib.reload(fever)
assert fever.RUNTIME_REVISION == 'colab-debug-v5'
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print('Debug runtime ready. The original ZIP and earlier experiment results were preserved.')

## 4. Training settings — already enabled
Two epochs per condition; learning rate 0.00002; weight decay 0.01; batch size 4
with 4-step gradient accumulation (effective batch 16); maximum 512 tokens;
seed 42; FP16. Validate and save every 1,000 optimizer steps, keeping the best
validation macro-F1 checkpoint and the latest resumable checkpoint. The session
budget is 180 minutes across conditions. No full benchmark evaluation runs here.

Training evidence is historical retrieved top-five sentence text for all labels,
including NEI. It is not newly fetched full-article RAG training data. Nonempty
retrieval does not guarantee that the evidence is sufficient. No improvement is claimed yet.

In [ ]:
EXPERIMENT_NAME = "roberta_full_unfreeze_last6_v1" # @param {type:"string"}
SESSION_MINUTES = 180 # @param {type:"integer"}
RUN_FULL_TRAINING = True # @param {type:"boolean"}
RUN_ROOT = DRIVE_PROJECT / "runs" / EXPERIMENT_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if not 1 <= SESSION_MINUTES <= 300: raise ValueError("Use a session budget between 1 and 300 minutes")
print("Training results and checkpoints:", RUN_ROOT)
print("Conditions: last 6 blocks plus the classification head.")

## 5. Verify the full training supplement
This checks file hashes, actual counts, all labels and normalized-text separation
between training, validation and official development before any training starts.

In [ ]:
EXPECTED_TRAINING = {'train_n': 116356, 'validation_n': 14534, 'files': {'train_full.jsonl': '2a3e6a7972446a8e89e6001611bd26ed08a4c7af28446351dfc05f4fdf220562', 'validation_full.jsonl': '19219943b93d34f0b37a8c15c44c7078e547b9af38eb193073fd8bea11881e56'}, 'scope': 'Full original training/validation after overlap exclusion; not the 96/96 pilot'}

training_archive = DRIVE_PROJECT / "FEVER_Training_Data.zip"
if not training_archive.is_file():
    raise FileNotFoundError("FEVER_Training_Data.zip is missing from FEVER_Project. Do not substitute pilot data.")
with zipfile.ZipFile(training_archive) as archive:
    expected_names = {"data/train_full.jsonl", "data/validation_full.jsonl", "training_ready.json", "preparation_protocol.json"}
    if set(archive.namelist()) != expected_names: raise ValueError("Unexpected supplement contents")
    ready = json.loads(archive.read("training_ready.json"))
    if ready != EXPECTED_TRAINING: raise ValueError("Supplement differs from the verified full training export")
    for name in ["train_full.jsonl", "validation_full.jsonl"]:
        content = archive.read("data/" + name)
        if hashlib.sha256(content).hexdigest() != ready["files"][name]:
            raise ValueError("Training file hash mismatch")
        target = PROJECT / "data" / name
        if target.exists() and target.read_bytes() != content:
            raise ValueError("Different full training data already exist")
        target.write_bytes(content)
train = fever.read_rows(PROJECT / "data/train_full.jsonl")
validation = fever.read_rows(PROJECT / "data/validation_full.jsonl")
assert len(train) == 116356 and len(validation) == 14534
assert len({r["id"] for r in train}) == len(train)
assert len({r["id"] for r in validation}) == len(validation)
assert all(r["label"] in fever.LABELS and isinstance(r["evidence_text"], str) for r in train+validation)
train_text = {fever.normalize(r["claim"]) for r in train}
validation_text = {fever.normalize(r["claim"]) for r in validation}
official_text = {fever.normalize(r["claim"]) for r in EVAL_ROWS}
if train_text & validation_text or (train_text | validation_text) & official_text:
    raise ValueError("Training/validation/official-development text overlap")
from collections import Counter
for label, records in [("Training",train),("Validation",validation)]:
    print(label, len(records), "claims; classes:", dict(Counter(r["label"] for r in records)))
    print("Empty evidence:", sum(not r["evidence_text"] for r in records))
print("PASS: all hashes/counts verified; zero normalized-text overlap across partitions.")
fever.write_once(RUN_ROOT / "data_verification.json", ready)
del train, validation, train_text, validation_text, official_text
BASELINE = {'scope': 'full', 'train_n': 116356, 'validation_n': 14534, 'train_sha256': '2a3e6a7972446a8e89e6001611bd26ed08a4c7af28446351dfc05f4fdf220562', 'validation_sha256': '19219943b93d34f0b37a8c15c44c7078e547b9af38eb193073fd8bea11881e56', 'runtime_sha256': '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06', 'initial_checkpoint': {'config.json': '6e879816747faab2f1f1577d7c3e687d7481691a9c4277ee93fbf56ca29f5686', 'merges.txt': '1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5', 'model.safetensors': '7a4a377205db4adf4af1671345e0ddbed2b164b78f1f4c1aaa13666b3b07e7f2', 'special_tokens_map.json': '06e405a36dfe4b9604f484f6a1e619af1a7f7d09e34a8555eb0b77b66318067f', 'tokenizer.json': '2c4c25f79756307e6e7e5bd219a0d602f784db9cdb63a9d76d8da7d05a032335', 'tokenizer_config.json': '4ea8648e4067230942ac6b32dd61328675bc99ae0a9f4cc665c30a95f0f68712', 'vocab.json': 'ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e'}, 'seed': 42, 'epochs': 2, 'learning_rate': 2e-05, 'max_length': 512, 'batch_size': 4, 'gradient_accumulation': 4, 'weight_decay': 0.01, 'evaluation_and_checkpoint_strategy': 'steps', 'checkpoint_steps': 1000, 'unfreezing': {'last_blocks': 3, 'classifier_head': True, 'trainable_parameters': 21856515, 'total_parameters': 124647939}, 'environment': {'packages': {'torch': '2.11.0+cu130', 'transformers': '4.57.6', 'accelerate': '1.15.0', 'bitsandbytes': '0.48.2', 'numpy': '2.4.6', 'scikit-learn': '1.6.1', 'faiss-cpu': '1.12.0'}, 'gpu': 'Tesla T4', 'cuda': '13.0'}}
for name, key in [("train_full.jsonl", "train_sha256"), ("validation_full.jsonl", "validation_sha256")]:
    if ready["files"][name] != BASELINE[key]:
        raise ValueError("Data do not match the last-3 baseline")
if fever.sha_file(PROJECT / "src/fever_colab_runtime.py") != BASELINE["runtime_sha256"]:
    raise ValueError("Training runtime differs from the last-3 baseline")
if fever.checkpoint_identity(PROJECT / "checkpoint/roberta_evidence") != BASELINE["initial_checkpoint"]:
    raise ValueError("Starting checkpoint differs from the last-3 baseline")
current_env = fever.environment()
if current_env != BASELINE["environment"]:
    print("Colab environment differs from the last-3 run. Both environments are recorded.")
    print("Last-3 environment:", BASELINE["environment"])
    print("Last-6 environment:", current_env)

## 6. Train / resume — enabled
The live bar shows optimizer steps. Logs include training loss, learning rate,
validation loss, validation macro-F1, GPU memory and checkpoint locations.
Every 20 seconds a heartbeat reports the current stage. A paused run is not a
completed experiment. Rerun this notebook to resume unchanged conditions.

In [ ]:
if not RUN_FULL_TRAINING:
    print("Training disabled.")
else:
    import time
    started = time.monotonic()
    for layers in [6]:
        remaining = SESSION_MINUTES - (time.monotonic() - started)/60
        if remaining < 1:
            print("Session budget reached. Run all again to resume remaining conditions.")
            break
        print("\nSTARTING / RESUMING: last", layers, "Transformer block(s).", flush=True)
        checkpoint = fever.finetune_roberta(PROJECT, RUN_ROOT, last_layers=layers,
            scope="full", minutes=remaining, checkpoint_steps=1000)
        if checkpoint is None:
            print("Paused and checkpointed. Run all again to resume.")
            break
        print("Completed checkpoint:", checkpoint, flush=True)

## 7. Saved progress and completion
This summarizes saved status after training finishes or pauses. Validation scores
choose checkpoints; they are not final test accuracy. Later we evaluate trained
checkpoints and the unchanged baseline on identical claims and evidence.

In [ ]:
for layers in [6]:
    name = "roberta_full_last" + str(layers)
    folder = RUN_ROOT / "training" / name
    status_path = folder / "monitor" / "status.json"
    print("\n", name)
    if status_path.is_file():
        status = json.loads(status_path.read_text())
        print(json.dumps({key:status.get(key) for key in ["status","stage","completed","total","reason","updated_utc"]}, indent=2))
    else:
        print("Not started yet.")
    if (folder / "complete.json").is_file():
        print("FINAL MODEL:", folder / "final")
        print((folder / "complete.json").read_text())
print("All losses and full logs are saved in Drive; the notebook's cell outputs also save there.")
if (folder / "complete.json").is_file():
    latest = max((folder / "checkpoints").glob("checkpoint-*/trainer_state.json"),
                 key=lambda path: int(path.parent.name.split("-")[-1]))
    state = json.loads(latest.read_text())
    score = state["best_metric"]
    print(f"Last 6 validation macro-F1: {score:.2%}")
    print("Last 3 validation macro-F1: 69.42%")
    print(f"Difference: {100 * (score - 0.6942348053196353):+.2f} percentage points")